In [1]:
from transformers import BertTokenizer
from text_preprocessing.tokenization import hf_token 
from text_preprocessing.embeddings import BertEmbedder
import torch
import torch.nn.functional as F
from training import ID2LABEL, TRAIN_CONFIG
from model import IntentClassifier, BANKING77_CONFIG

d:\@FYP-IntentClassifier\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = IntentClassifier(
    cfg=BANKING77_CONFIG,
    num_classes=TRAIN_CONFIG["num_classes"],
    pool=TRAIN_CONFIG["pool"],
).to(device)
model.load_state_dict(torch.load('best_model.pt', weights_only=True))

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased', token=hf_token)
embedder = BertEmbedder(device=device)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 445.66it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:

def predict_intent(text, model, device, id2label, threshold=0.4, max_length=64):
    model.eval()

    # ── Step 1: Tokenize ─────────────────────────────────────────────────────
    tokens = tokenizer(
        text,
        max_length=max_length,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    # ── Step 2: Get BERT embeddings ──────────────────────────────────────────
    emb = embedder.get_embeddings(tokens['input_ids'], tokens['attention_mask'])

    # ── Step 3: Classify ─────────────────────────────────────────────────────
    mask = tokens['attention_mask'].to(device)

    with torch.no_grad():
        logits = model(emb, mask)
        probs = F.softmax(logits, dim=1)

    top_probs, top_ids = probs.topk(3, dim=1)

    print(f"Input: '{text}'")
    print(f"{'Rank':<6} {'Intent':<35} {'Confidence'}")
    print("-" * 55)
    for rank, (prob, idx) in enumerate(zip(top_probs[0], top_ids[0]), 1):
        flag = "NO!" if prob.item() < threshold else ""
        print(f"  {rank:<4} {id2label[idx.item()]:<35} {prob.item():.2%}  {flag}")

    top_label = id2label[top_ids[0][0].item()]
    top_conf = top_probs[0][0].item()
    return top_label, top_conf




In [11]:
predict_intent('''Hey I tried sending money yesterday and it looked successful but my friend says nothing arrived yet, can you check?''', model, device, ID2LABEL)

Input: 'Hey I tried sending money yesterday and it looked successful but my friend says nothing arrived yet, can you check?'
Rank   Intent                              Confidence
-------------------------------------------------------
  1    transfer_not_received_by_recipient  99.37%  
  2    balance_not_updated_after_cheque_or_cash_deposit 0.13%  NO!
  3    Refund_not_showing_up               0.05%  NO!


('transfer_not_received_by_recipient', 0.9936890602111816)